# Longitudinal Exploration

Converted from R Markdown to Python Jupyter Notebook

## Import Libraries and Load Data

In [ ]:
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

from utils import (
    read_clean_sheds,
    build_car_history,
    analyze_ev_ownership_data,
    save_plot,
)

current_dir = Path.cwd()

_root = current_dir
while not (_root / "config.yaml").exists() and _root != _root.parent:
    _root = _root.parent
with open(_root / "config.yaml") as f:
    config = yaml.safe_load(f)

data_dir = Path(config["paths"]["data_dir"])
sheds_files = {int(k): v for k, v in config["sheds_files"].items()}

years = [2016, 2017, 2018, 2019, 2020, 2021, 2023, 2025]

waves = {}
for year in years:
    file_path = data_dir / sheds_files[year]
    wave_name = f"w{year}"
    print(f"Reading {year} ...")
    waves[wave_name], _ = read_clean_sheds(str(file_path))
    print(f"{len(waves[wave_name])} valid respondents")

In [ ]:
# Find total unique IDs across all waves
all_ids = pd.concat([df[['id']] for df in waves.values()])['id'].unique()

print(f"\nTotal unique IDs across all waves: {len(all_ids)}")

# Summary
total_respondents = sum(len(df) for df in waves.values())

print(f"\n=== Summary ===")
print(f"Total respondents (all waves): {total_respondents}")
print(f"Total unique IDs: {len(all_ids)}")
print(f"Duplicate appearances: {total_respondents - len(all_ids)}")

# To see how many times each ID appears
id_list = []
for wave_name, df in waves.items():
    for id_val in df['id']:
        id_list.append({'wave': wave_name, 'id': id_val})

id_df = pd.DataFrame(id_list)
id_counts = id_df.groupby('id').size().reset_index(name='n').sort_values('n', ascending=False)

print(f"\nDistribution of appearances:")
print(id_counts['n'].value_counts().sort_index())

## Build Car History to See EV Ownership Over Time

In [ ]:
# Convert waves dict keys from "w2016" to "2016"
all_waves = {key.replace('w', ''): df for key, df in waves.items()}

# Build car history
car_history = build_car_history(all_waves)
print(car_history)

# Save to CSV
car_history.to_csv("car_history.csv", index=False)
# Check what's actually in mob3_3 across all waves
print("=== mob3_3 value distribution ===")
print(car_history['mob3_3'].value_counts(dropna=False).head(20))

print("\n=== How many valid mob3_3 values (> 0) per year? ===")
valid_mob3_3 = car_history[car_history['mob3_3'] > 0]
print(valid_mob3_3.groupby('year_wave').size())

print("\n=== Check one wave's original data ===")
# Check 2025 wave directly
w2025 = waves['w2025']
print(f"mob3_3 in 2025 wave:")
print(w2025['mob3_3'].value_counts(dropna=False).head(20))

In [ ]:
# Run analysis with car history for each year
results_list = []
for year in [2019, 2020, 2021, 2023, 2025]:
    result = analyze_ev_ownership_data(car_history, year)
    results_list.append(result)

results = pd.concat(results_list, ignore_index=True)
print(results)

In [ ]:
# Print summary table
summary_cols = ['year', 'n_total', 'n_car_owners', 'n_ev_main', 'n_ev_secondary',
                'n_ev_total', 'n_hybrid_gas', 'n_hybrid_diesel', 'ev_rate_car_owners']

summary_df = results[summary_cols].copy()
summary_df['ev_rate_car_owners'] = summary_df['ev_rate_car_owners'].apply(
    lambda x: f"{x:.2%}" if pd.notna(x) else "N/A"
)

# Rename columns for display
summary_df.columns = ['Year', 'N', 'Car Own.', 'EV Main', 'EV 2nd',
                      'EV Total', 'Hybrid Gas', 'Hybrid Dies.', 'EV Rate']

print(summary_df.to_string(index=False))

## Visualizations

In [ ]:
# Prepare data for stacked bars
plot_data = results[results['year'] >= 2019].copy()
plot_data['n_hybrid_total'] = plot_data['n_hybrid_gas'] + plot_data['n_hybrid_diesel']
plot_data['total_electrified'] = plot_data['n_ev_total'] + plot_data['n_hybrid_total']
plot_data['total_pct'] = plot_data['total_electrified'] / plot_data['n_car_owners']

# Reshape for plotting
p2_data = plot_data[['year', 'n_ev_total', 'n_hybrid_total', 'n_car_owners', 'total_pct']].melt(
    id_vars=['year', 'n_car_owners', 'total_pct'],
    value_vars=['n_ev_total', 'n_hybrid_total'],
    var_name='type',
    value_name='count'
)

# Rename types
type_mapping = {
    'n_ev_total': 'Pure Electric',
    'n_hybrid_total': 'Hybrid'
}
p2_data['type'] = p2_data['type'].map(type_mapping)
p2_data['percentage'] = p2_data['count'] / p2_data['n_car_owners']

# Totals data
totals_data = plot_data[['year', 'total_pct']].drop_duplicates()

print(p2_data)

In [ ]:
# Simple line comparison EV vs Hybrid
line_data = results[results['year'] >= 2019].copy()
line_data['n_hybrid_total'] = line_data['n_hybrid_gas'] + line_data['n_hybrid_diesel']

# Colors
colors = {
    'Pure Electric': '#1B5E20',
    'Hybrid': '#FF9800'
}

fig, ax = plt.subplots(figsize=(10, 6))

# Plot Pure Electric
ax.plot(line_data['year'], line_data['n_ev_total'], 
        marker='o', linewidth=2, markersize=8,
        color=colors['Pure Electric'], label='Pure Electric')

# Plot Hybrid
ax.plot(line_data['year'], line_data['n_hybrid_total'],
        marker='o', linewidth=2, markersize=8,
        color=colors['Hybrid'], label='Hybrid')

# Add value labels
for _, row in line_data.iterrows():
    ax.annotate(f"{int(row['n_ev_total'])}", 
                (row['year'], row['n_ev_total']),
                textcoords="offset points", xytext=(0, 10),
                ha='center', fontsize=9, color=colors['Pure Electric'])
    ax.annotate(f"{int(row['n_hybrid_total'])}",
                (row['year'], row['n_hybrid_total']),
                textcoords="offset points", xytext=(0, 10),
                ha='center', fontsize=9, color=colors['Hybrid'])

ax.set_xticks(line_data['year'].unique())
ax.set_title('Pure Electric vs Hybrid Adoption Trends', fontweight='bold', fontsize=12)
ax.set_xlabel('')
ax.set_ylabel('Number of Households')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()

# Save plot
figures_dir = current_dir / "figures"
save_plot(fig, path=str(figures_dir), filename="ev_vs_hybrid_trend")

plt.show()

## Summary Statistics

In [ ]:
# EV Growth statistics
results_clean = results[results['ev_rate_car_owners'].notna()].copy()

rate_2019 = results_clean.loc[results_clean['year'] == 2019, 'ev_rate_car_owners'].values[0]
rate_2025 = results_clean.loc[results_clean['year'] == 2025, 'ev_rate_car_owners'].values[0]

print("EV GROWTH:")
print(f"  2019: {rate_2019:.2%}")
print(f"  2025: {rate_2025:.2%}")
print(f"  Growth: {rate_2025 / rate_2019:.1f}x\n")

res_2025 = results[results['year'] == 2025].iloc[0]
print(f"  Total EVs: {int(res_2025['n_ev_total'])}")
print(f"  Main car EVs: {int(res_2025['n_ev_main'])}")
print(f"  Secondary EVs: {int(res_2025['n_ev_secondary'])}")
print(f"  Total Hybrids: {int(res_2025['n_hybrid_gas'] + res_2025['n_hybrid_diesel'])}")

## Additional Visualizations

In [ ]:
# Stacked bar chart showing EV and Hybrid composition
fig, ax = plt.subplots(figsize=(10, 6))

years_plot = line_data['year'].values
ev_counts = line_data['n_ev_total'].values
hybrid_counts = line_data['n_hybrid_total'].values

bar_width = 0.6

# Stacked bars
ax.bar(years_plot, hybrid_counts, bar_width, label='Hybrid', color=colors['Hybrid'])
ax.bar(years_plot, ev_counts, bar_width, bottom=hybrid_counts, 
       label='Pure Electric', color=colors['Pure Electric'])

# Add total labels on top
for year, ev, hybrid in zip(years_plot, ev_counts, hybrid_counts):
    total = ev + hybrid
    ax.annotate(f'{int(total)}', (year, total), 
                textcoords="offset points", xytext=(0, 5),
                ha='center', fontweight='bold')

ax.set_xticks(years_plot)
ax.set_title('Total Electrified Vehicles (EV + Hybrid)', fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Households')
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# EV adoption rate over time
fig, ax = plt.subplots(figsize=(10, 6))

rate_data = results_clean[['year', 'ev_rate_car_owners']].copy()

ax.plot(rate_data['year'], rate_data['ev_rate_car_owners'] * 100,
        marker='o', linewidth=2, markersize=10, color='#1B5E20')

# Add percentage labels
for _, row in rate_data.iterrows():
    ax.annotate(f"{row['ev_rate_car_owners']:.1%}",
                (row['year'], row['ev_rate_car_owners'] * 100),
                textcoords="offset points", xytext=(0, 12),
                ha='center', fontsize=10, fontweight='bold')

ax.set_xticks(rate_data['year'])
ax.set_title('EV Adoption Rate Among Car Owners', fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('EV Adoption Rate (%)')
ax.set_ylim(0, max(rate_data['ev_rate_car_owners'] * 100) * 1.3)
ax.grid(True, alpha=0.3)

plt.tight_layout()
save_plot(fig, path=str(figures_dir), filename="ev_adoption_rate_trend")
plt.show()

In [ ]:
# Respondent participation across waves
participation_dist = id_counts['n'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(participation_dist.index, participation_dist.values, color='steelblue')

# Add count labels
for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{int(height):,}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Number of Waves Participated')
ax.set_ylabel('Number of Respondents')
ax.set_title('Respondent Participation Across Survey Waves', fontweight='bold')
ax.set_xticks(participation_dist.index)

plt.tight_layout()
plt.show()

# Print statistics
print(f"\nParticipation Statistics:")
print(f"  Single wave only: {participation_dist.get(1, 0):,} respondents")
print(f"  All {len(years)} waves: {participation_dist.get(len(years), 0):,} respondents")
print(f"  Average waves per respondent: {id_counts['n'].mean():.2f}")

## EV Ownership Over Time — md_708 and Car History

In [ ]:
md_708_summary_rows = []
for wave_name, df in waves.items():
    year = int(wave_name.replace('w', ''))
    n_total = len(df)

    n_car_owners_md220 = (
        ((df['md_220'].astype(float) >= 1) & (df['md_220'].astype(float) < 90)).sum()
        if 'md_220' in df.columns else None
    )
    n_ev_md708 = (df['md_708'] == 1).sum() if 'md_708' in df.columns else None
    n_ev_missing = df['md_708'].isna().sum() if 'md_708' in df.columns else None
    n_total_vehicles_md220 = (
        df.loc[(df['md_220'].astype(float) >= 1) & (df['md_220'].astype(float) < 90), 'md_220']
          .astype(float).sum()
        if 'md_220' in df.columns else None
    )

    md_708_summary_rows.append({
        'year': year,
        'n_total': n_total,
        'n_car_owners_md220': n_car_owners_md220,
        'n_ev_md708': n_ev_md708,
        'n_ev_missing': n_ev_missing,
        'n_total_vehicles_md220': n_total_vehicles_md220,
        'ev_rate_md708': n_ev_md708 / n_total if n_ev_md708 is not None else None,
        'ev_rate_car_owners': n_ev_md708 / n_car_owners_md220 if (n_ev_md708 and n_car_owners_md220) else None,
        'ev_rate_vehicles': n_ev_md708 / n_total_vehicles_md220 if (n_ev_md708 and n_total_vehicles_md220) else None,
    })

md_708_summary = pd.DataFrame(md_708_summary_rows).sort_values('year')
print(md_708_summary.to_string(index=False))

## Build Car History (Stata Logic) and Per-Wave Stats

In [ ]:
# waves dict has keys like 'w2016' — strip prefix for build_car_history
all_waves = {k.replace('w', ''): v for k, v in waves.items()}
car_history = build_car_history(all_waves)

results_rows = []
for year in [2016, 2017, 2018, 2019, 2020, 2021, 2023, 2025]:
    yr_str = str(year)
    df_yr  = car_history[car_history['year_wave'] == year]
    raw    = all_waves[yr_str]

    car_owners_mask = (df_yr['mob2_1'] > 0) & (df_yr['mob2_1'] < 90)
    car_owners_df   = df_yr[car_owners_mask]

    n_total_vehicles_mob2 = (
        raw.loc[(raw['mob2_1'] > 0) & (raw['mob2_1'] < 90), 'mob2_1'].sum()
        if 'mob2_1' in raw.columns else None
    )
    n_total_vehicles_md220 = (
        raw.loc[(raw['md_220'].astype(float) > 0) & (raw['md_220'].astype(float) < 90), 'md_220']
          .astype(float).sum()
        if 'md_220' in raw.columns else None
    )

    n_elec_total  = ((car_owners_df['mob3_3_filled'] == 8) | (car_owners_df.get('mob2_e', pd.Series(dtype=float)) == 1)).sum() if 'mob3_3_filled' in car_owners_df.columns else 0
    n_hybrid_total = car_owners_df['mob3_3_filled'].isin([5, 6, 7]).sum() if 'mob3_3_filled' in car_owners_df.columns else 0

    results_rows.append({
        'year': year,
        'n_total': len(df_yr),
        'n_car_owners': car_owners_mask.sum(),
        'n_elec_total': n_elec_total,
        'n_hybrid_total': n_hybrid_total,
        'n_total_vehicles_mob2': n_total_vehicles_mob2,
        'n_total_vehicles_md220': n_total_vehicles_md220,
        'ev_rate_vehicles': n_elec_total / n_total_vehicles_mob2 * 100 if n_total_vehicles_mob2 else None,
    })

results_stata = pd.DataFrame(results_rows)
print(results_stata.to_string(index=False))

## FSO (BFS) Official Vehicle Statistics — API Call

In [ ]:
import requests

url = (
    "https://disseminate.stats.swiss/rest/data/CH1.MFZ_IVS,DF_MFZ_1_EMISSION,1.0.0/"
    "_T._T._T._T._T+PC+PH+DC+DH+HP+HD+EL+FC+GA+_O._T._T.A"
    "?startPeriod=2005&endPeriod=2025"
    "&dimensionAtObservation=AllDimensions&format=csvfilewithlabels"
)

resp = requests.get(url)
print("Status:", resp.status_code)

from io import StringIO
fso_raw = pd.read_csv(StringIO(resp.text))
print("Columns:", fso_raw.columns.tolist())
print(fso_raw.head(3))

In [ ]:
# FSO fuel type codes (UV_RV_FUEL column) → internal names
FUEL_CODE_MAP = {
    '_T': 'total',
    'EL': 'bev',
    'HP': 'phev_petrol',
    'HD': 'phev_diesel',
    'PH': 'petrol_hev',
    'DH': 'diesel_hev',
    'PC': 'petrol',
    'DC': 'diesel',
    'FC': 'fcev',
    'GA': 'gas',
    '_O': 'other',
}

# Detect column names (robust to minor API changes)
code_col  = next(c for c in fso_raw.columns if c.upper() == 'UV_RV_FUEL')
time_col  = next(c for c in fso_raw.columns if 'TIME_PERIOD' in c.upper())
value_col = next(c for c in fso_raw.columns if 'OBS_VALUE' in c.upper())

fso = (
    fso_raw[[code_col, time_col, value_col]]
    .rename(columns={code_col: 'fuel_code', time_col: 'year', value_col: 'value'})
    .dropna(subset=['year'])
    .assign(fuel_code=lambda d: d['fuel_code'].map(FUEL_CODE_MAP).fillna('other'),
            year=lambda d: d['year'].astype(int),
            value=lambda d: pd.to_numeric(d['value'], errors='coerce'))
    .groupby(['year', 'fuel_code'])['value'].sum()
    .unstack('fuel_code', fill_value=0)
    .reset_index()
)

fso['phev']         = fso.get('phev_petrol', 0) + fso.get('phev_diesel', 0)
fso['hev']          = fso.get('petrol_hev',  0) + fso.get('diesel_hev',  0)
fso['total_hybrid'] = fso['phev'] + fso['hev']
fso['ev_pct']       = fso['bev']          / fso['total'] * 100
fso['hybrid_pct']   = fso['total_hybrid'] / fso['total'] * 100

print(fso[['year', 'bev', 'total_hybrid', 'total', 'ev_pct', 'hybrid_pct']].to_string(index=False))

## Comparison: SHEDS vs FSO Official Statistics

In [ ]:
analysis_years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

# FSO long format
fso_long = (
    fso[fso['year'].isin(analysis_years)][['year', 'ev_pct', 'hybrid_pct']]
    .melt(id_vars='year', var_name='type', value_name='percentage')
    .assign(type=lambda d: d['type'].map({'ev_pct': 'Pure Electric', 'hybrid_pct': 'Hybrid'}),
            source='FSO (Official)')
)

# SHEDS history (mob2/mob3 based)
sheds_history_rows = []
for _, row in results_stata.iterrows():
    year = int(row['year'])
    veh  = row['n_total_vehicles_mob2']
    if veh and veh > 0:
        if year in [2016, 2017, 2018]:
            sheds_history_rows.append({'year': year, 'type': 'EV+Hybrid (mixed)',
                                       'percentage': row['n_elec_total'] / veh * 100,
                                       'source': 'SHEDS (History)'})
        else:
            sheds_history_rows.append({'year': year, 'type': 'Pure Electric',
                                       'percentage': row['n_elec_total'] / veh * 100,
                                       'source': 'SHEDS (History)'})
            sheds_history_rows.append({'year': year, 'type': 'Hybrid',
                                       'percentage': row['n_hybrid_total'] / veh * 100,
                                       'source': 'SHEDS (History)'})

sheds_history_pop = pd.DataFrame(sheds_history_rows)

# SHEDS md_708 (Intervista EV flag)
sheds_md708_pop = (
    md_708_summary[md_708_summary['year'].isin(analysis_years)]
    .assign(percentage=lambda d: d['n_ev_md708'] / d['n_total_vehicles_md220'] * 100,
            type='Pure Electric',
            source='SHEDS (md_708)')
    [['year', 'type', 'percentage', 'source']]
)

print("FSO sample:"); print(fso_long.head())
print("SHEDS history sample:"); print(sheds_history_pop.head())
print("SHEDS md_708 sample:"); print(sheds_md708_pop.head())

## Plot: SHEDS vs FSO (with md_708)

In [ ]:
COLORS = {'Hybrid': '#FF9800', 'Pure Electric': '#1B5E20', 'EV+Hybrid (mixed)': '#66BB6A'}
TYPE_ORDER   = ['Hybrid', 'Pure Electric', 'EV+Hybrid (mixed)']
SOURCE_ORDER = ['FSO (Official)', 'SHEDS (History)', 'SHEDS (md_708)']

comparison_data = (
    pd.concat([fso_long, sheds_history_pop, sheds_md708_pop], ignore_index=True)
    .assign(source=lambda d: pd.Categorical(d['source'], categories=SOURCE_ORDER, ordered=True),
            type=lambda d: pd.Categorical(d['type'],   categories=TYPE_ORDER,   ordered=True))
)

years_plot = sorted(comparison_data['year'].unique())
fig, axes = plt.subplots(1, len(years_plot), figsize=(30, 6), sharey=True)

for ax, year in zip(axes, years_plot):
    yr_data = comparison_data[comparison_data['year'] == year]
    totals  = yr_data.groupby('source', observed=True)['percentage'].sum()

    bottoms = {s: 0.0 for s in SOURCE_ORDER}
    x_pos   = {s: i for i, s in enumerate(SOURCE_ORDER)}

    for type_val in TYPE_ORDER:
        for source in SOURCE_ORDER:
            val = yr_data[(yr_data['source'] == source) & (yr_data['type'] == type_val)]['percentage']
            if val.empty: continue
            v = float(val.iloc[0])
            ax.bar(x_pos[source], v, bottom=bottoms[source], color=COLORS[type_val],
                   width=0.5, label=type_val if (year == years_plot[0] and bottoms[source] == 0) else '')
            if v > 0.3:
                ax.text(x_pos[source], bottoms[source] + v / 2, f'{v:.1f}%',
                        ha='center', va='center', fontsize=7.5, color='white', fontweight='bold')
            bottoms[source] += v

    for source in SOURCE_ORDER:
        tot = totals.get(source, 0)
        if tot > 0:
            ax.text(x_pos[source], tot + 0.15, f'{tot:.1f}%',
                    ha='center', va='bottom', fontsize=8, fontweight='bold')

    ax.set_title(str(year), fontweight='bold', fontsize=12)
    ax.set_xticks(list(x_pos.values()))
    ax.set_xticklabels(list(x_pos.keys()), rotation=15, ha='right', fontsize=8)
    ax.set_xlabel('')
    for spine in ['top', 'right']: ax.spines[spine].set_visible(False)

axes[0].set_ylabel('Percentage of total vehicles (%)', fontsize=10, fontweight='bold')

# Shared legend
handles = [plt.Rectangle((0,0),1,1, color=COLORS[t]) for t in TYPE_ORDER]
fig.legend(handles, TYPE_ORDER, loc='lower center', ncol=3, fontsize=11,
           bbox_to_anchor=(0.5, -0.08), frameon=False)

fig.tight_layout()
plt.show()
save_plot(fig, path='plots', filename='bfs_vs_sheds_comparison_with_md708', width=24, height=6)

## Plot: SHEDS vs FSO (without md_708)

In [ ]:
comparison_data_no708 = (
    pd.concat([fso_long, sheds_history_pop], ignore_index=True)
    .assign(source=lambda d: pd.Categorical(d['source'],
                                            categories=['FSO (Official)', 'SHEDS (History)'],
                                            ordered=True),
            type=lambda d: pd.Categorical(d['type'], categories=TYPE_ORDER, ordered=True))
)

SOURCE_ORDER_2 = ['FSO (Official)', 'SHEDS (History)']
years_plot2    = sorted(comparison_data_no708['year'].unique())
fig2, axes2    = plt.subplots(1, len(years_plot2), figsize=(30, 6), sharey=True)

for ax, year in zip(axes2, years_plot2):
    yr_data = comparison_data_no708[comparison_data_no708['year'] == year]
    totals  = yr_data.groupby('source', observed=True)['percentage'].sum()
    bottoms = {s: 0.0 for s in SOURCE_ORDER_2}
    x_pos   = {s: i for i, s in enumerate(SOURCE_ORDER_2)}

    for type_val in TYPE_ORDER:
        for source in SOURCE_ORDER_2:
            val = yr_data[(yr_data['source'] == source) & (yr_data['type'] == type_val)]['percentage']
            if val.empty: continue
            v = float(val.iloc[0])
            ax.bar(x_pos[source], v, bottom=bottoms[source], color=COLORS[type_val], width=0.4)
            if v > 0.3:
                ax.text(x_pos[source], bottoms[source] + v / 2, f'{v:.1f}%',
                        ha='center', va='center', fontsize=8, color='white', fontweight='bold')
            bottoms[source] += v

    for source in SOURCE_ORDER_2:
        tot = totals.get(source, 0)
        if tot > 0:
            ax.text(x_pos[source], tot + 0.15, f'{tot:.1f}%',
                    ha='center', va='bottom', fontsize=8.5, fontweight='bold')

    ax.set_title(str(year), fontweight='bold', fontsize=12)
    ax.set_xticks(list(x_pos.values()))
    ax.set_xticklabels(list(x_pos.keys()), rotation=15, ha='right', fontsize=9)
    for spine in ['top', 'right']: ax.spines[spine].set_visible(False)

axes2[0].set_ylabel('Percentage of total vehicles (%)', fontsize=10, fontweight='bold')

handles2 = [plt.Rectangle((0,0),1,1, color=COLORS[t]) for t in TYPE_ORDER]
fig2.legend(handles2, TYPE_ORDER, loc='lower center', ncol=3, fontsize=11,
            bbox_to_anchor=(0.5, -0.08), frameon=False)

fig2.tight_layout()
plt.show()
save_plot(fig2, path='plots', filename='bfs_vs_sheds_comparison_without_md708', width=24, height=6)